# METER NYU Depth v2 Kaggle Training

Concise training notebook for the METER paper reproduction. Depth is converted from meters to centimeters; depth `<= 1 cm` is masked from training loss and metrics. Most logic lives in importable `.py` files so the same code can be zipped and uploaded with Kaggle Model weights.

In [ ]:
# Optional on Kaggle if packages are missing.
# !pip install -q wandb h5py

In [ ]:
from __future__ import annotations

import json
import random
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

In [ ]:
@dataclass(frozen=True)
class NotebookConfig:
    kaggle_dataset_slug: str = "your-nyu-depth-v2-slug"
    local_data_root: str = "../nyu_depth_v2"
    local_code_root: str = ".."
    arch_type: str = "s"
    input_height: int = 192
    input_width: int = 256
    max_depth_cm: float = 1000.0
    invalid_depth_threshold_cm: float = 1.0
    num_epochs: int = 60
    batch_size: int = 256
    num_workers: int = 4
    prefetch_factor: int = 1
    pin_memory: bool = True
    persistent_workers: bool = True
    learning_rate: float = 1e-3
    adam_beta1: float = 0.9
    adam_beta2: float = 0.999
    weight_decay: float = 0.01
    scheduler_step_size: int = 20
    scheduler_gamma: float = 0.1
    use_amp: bool = True
    seed: int = 42
    lambda_1: float = 0.5
    lambda_2: float = 100.0
    lambda_3: float = 100.0
    flip: float = 0.5
    mirror: float = 0.5
    c_swap: float = 0.5
    random_crop: float = 1.0
    shifting_strategy: float = 0.5
    color_low: float = 0.9
    color_high: float = 1.1
    depth_shift_min_cm: float = -10.0
    depth_shift_max_cm: float = 10.0
    max_train_samples: int | None = None
    max_val_samples: int | None = None
    output_dir: str = "outputs"
    use_wandb: bool = True
    wandb_project: str = "meter-nyu-depth"
    wandb_run_name: str = "meter-nyu-s-cm"
    wandb_mode: str = "online"
    vis_every_epochs: int = 10
    vis_num_samples: int = 4
    checkpoint_name: str = "meter_nyu_best.pt"
    final_checkpoint_name: str = "meter_nyu_final.pt"

CFG = NotebookConfig()

In [ ]:
def running_on_kaggle() -> bool:
    return Path("/kaggle/input").exists()

def resolve_code_root() -> Path:
    candidates = []
    if running_on_kaggle():
        base = Path("/kaggle/input") / CFG.kaggle_dataset_slug
        candidates.extend([base, base / "METER", Path("/kaggle/working")])
    candidates.append(Path(CFG.local_code_root).resolve())
    for path in candidates:
        if (path / "architecture.py").exists():
            return path
    raise FileNotFoundError(f"Could not find architecture.py in {candidates}")

CODE_ROOT = resolve_code_root()
sys.path.insert(0, str(CODE_ROOT))
print("CODE_ROOT =", CODE_ROOT)

In [ ]:
from architecture import build_METER_model
from meter_data import DataConfig, NYUH5DepthDataset, discover_h5_files, resolve_data_root
from meter_loss import BalancedMETERLoss, LossConfig
from meter_train import TrainConfig, fit, seed_everything

In [ ]:
seed_everything(CFG.seed)
random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_root = resolve_data_root(CFG.kaggle_dataset_slug, CFG.local_data_root)
print("DATA_ROOT =", data_root)
print("DEVICE =", device)
print(json.dumps(asdict(CFG), indent=2))

In [ ]:
data_config = DataConfig(
    input_height=CFG.input_height,
    input_width=CFG.input_width,
    max_depth_cm=CFG.max_depth_cm,
    invalid_depth_threshold_cm=CFG.invalid_depth_threshold_cm,
    flip=CFG.flip,
    mirror=CFG.mirror,
    c_swap=CFG.c_swap,
    random_crop=CFG.random_crop,
    shifting_strategy=CFG.shifting_strategy,
    color_low=CFG.color_low,
    color_high=CFG.color_high,
    depth_shift_min_cm=CFG.depth_shift_min_cm,
    depth_shift_max_cm=CFG.depth_shift_max_cm,
)
train_files = discover_h5_files(data_root, "train", CFG.max_train_samples)
val_files = discover_h5_files(data_root, "val", CFG.max_val_samples)
print(f"train={len(train_files):,}, val={len(val_files):,}")

train_ds = NYUH5DepthDataset(train_files, data_config, train=True)
val_ds = NYUH5DepthDataset(val_files, data_config, train=False)
loader_kwargs = {
    "batch_size": CFG.batch_size,
    "num_workers": CFG.num_workers,
    "pin_memory": CFG.pin_memory and device.type == "cuda",
    "persistent_workers": CFG.persistent_workers and CFG.num_workers > 0,
}
if CFG.num_workers > 0:
    loader_kwargs["prefetch_factor"] = CFG.prefetch_factor
train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
batch = next(iter(train_loader))
print(batch["image"].shape, batch["depth"].shape, batch["mask"].shape, "valid:", int(batch["mask"].sum()))

In [ ]:
model = build_METER_model(device=str(device), arch_type=CFG.arch_type).to(device)
criterion = BalancedMETERLoss(LossConfig(
    max_depth_cm=CFG.max_depth_cm,
    invalid_depth_threshold_cm=CFG.invalid_depth_threshold_cm,
    lambda_1=CFG.lambda_1,
    lambda_2=CFG.lambda_2,
    lambda_3=CFG.lambda_3,
)).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    betas=(CFG.adam_beta1, CFG.adam_beta2),
    weight_decay=CFG.weight_decay,
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=CFG.scheduler_step_size, gamma=CFG.scheduler_gamma)
with torch.no_grad():
    dummy = torch.zeros(1, 3, CFG.input_height, CFG.input_width, device=device)
    print("dummy output:", tuple(model(dummy).shape))

In [ ]:
train_config = TrainConfig(
    num_epochs=CFG.num_epochs,
    learning_rate=CFG.learning_rate,
    adam_beta1=CFG.adam_beta1,
    adam_beta2=CFG.adam_beta2,
    weight_decay=CFG.weight_decay,
    scheduler_step_size=CFG.scheduler_step_size,
    scheduler_gamma=CFG.scheduler_gamma,
    use_amp=CFG.use_amp,
    max_depth_cm=CFG.max_depth_cm,
    invalid_depth_threshold_cm=CFG.invalid_depth_threshold_cm,
    output_dir=CFG.output_dir,
    checkpoint_name=CFG.checkpoint_name,
    final_checkpoint_name=CFG.final_checkpoint_name,
    use_wandb=CFG.use_wandb,
    wandb_project=CFG.wandb_project,
    wandb_run_name=CFG.wandb_run_name,
    wandb_mode=CFG.wandb_mode,
    vis_every_epochs=CFG.vis_every_epochs,
    vis_num_samples=CFG.vis_num_samples,
)
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    train_config=train_config,
    experiment_config=asdict(CFG),
)

## Package for Kaggle Model

Run this from the repository root after training to create a local zip you can upload manually to Kaggle Model:

```bash
conda run -n mlx python scripts/package_kaggle_model.py   --checkpoint-dir outputs/checkpoints   --output-zip outputs/kaggle_meter_model.zip   --arch-type s
```

Optional API upload, only if your Kaggle credentials are configured:

```bash
conda run -n mlx python scripts/package_kaggle_model.py   --checkpoint-dir outputs/checkpoints   --output-zip outputs/kaggle_meter_model.zip   --arch-type s   --upload   --kaggle-model username/meter-nyu
```